# ReadMe

**NOTICE** Ziyi Ding, the adaptor of the present code. 2024.9.9 edited.

1. 将名为exp_target_words.txt的文件置于与代码相同的文件夹内，此物乃chatGPT的提示词表；
2. 修改Settings内的各项参数
3. 输入OpenAI的API，Tokens数量按照“提问+回答”字数付费；
4. 修改指导语，对当前指导语使用类比思维；
5. 如果是第一次运行，需要安装名为numpy和OpenAI的package；

## OpenAI 参考资料
官方manual：https://platform.openai.com/docs/guides/text-generation/chat-completions-api   
OpenAI付费方案：https://openai.com/pricing   
不会设置参数看这里：https://platform.openai.com/docs/guides/text-generation/parameter-details   
在线调试参数和指导语：https://platform.openai.com/playground?mode=complete

In [1]:
## Packages
import numpy as np
import random
from openai import OpenAI
# To install packages in Python: 
    # Open Anaconda Powershell Prompt, 
    # Input the order: pip install [name of a package]

In [2]:
## Settings
# Experiment Settings
RepeatTime = 120 # 数字代表每个提示词会被chatGPT回答几次
AnswerTime = 41720 # ChatGPT需要回答多少次
WordOnce = 1 # chatGPT一次回答多少个提示词

# OpenAI Parameters

MyAPI = "APIkey" # OpenAI官网购买的API，按需购买Tokens数量，Tokens数量按照“提问+回答”字数付费
MyModel = "gpt-4o-2024-08-06" # 调用OpenAI的哪个模型
MyTemperature = 1.5 # 0-2, 越小回答的多样性越小
MyMaxTokens = 100 # 一次“提问+回答”的最大字数上限，避免不知不觉破产
MyFreqPenalty = 1 # 合理值0-1, 越高重复性越少，重复性是相对于一次“提问+回答”而言的

client = OpenAI(api_key = MyAPI)

In [3]:
## Test API
response = client.chat.completions.create(
   model="gpt-4o-2024-08-06",
   messages=[{"role": "user", 
               "content": "鲁迅和周树人是什么关系？"}])
print(response.choices[0].message.content)

鲁迅是周树人的笔名。周树人是鲁迅的本名，两者是同一个人。鲁迅是中国著名的文学家、思想家和革命家，他的作品对中国近现代文化产生了深远的影响。


In [4]:
## Input the cue words
with open("./target_words.txt", "r", encoding = "utf-8") as file:
    # file - 指代open命令打开的文件，此处是target_words.txt
    # "r" - 读入文件内容，当前txt文件内的词按照回车键分割
    # encoding = "utf-8" - txt文件内的词按照utf-8的编码方式读入
    data = file.read()
    CUES = [i for i in data.split("\n") if i != ""] # []是创建列表，接近MATLAB的cell
    # 除去空元素外，按照回车键分割CUES中的i个元素

print(len(CUES))
print(CUES[0:WordOnce]) # 查看CUES的前几个元素

164
['专家']


In [5]:
## Randomize items in the CUES
CUES = [j for i in [random.sample(CUES, len(CUES)) for i in range(RepeatTime)] for j in i] 
# 随机排列CUES[RepeatTime]次，使得CUES中元素数量变为原数量的[RepeatTime]倍

CUES_all = CUES

print(len(CUES_all))
print(CUES_all[0:3]) # 查看CUES_all的前10个元素

19680
['位置', '安全', '姐妹']


In [6]:
## Set functions getting access to OpenAI
import time, warnings, pickle, csv

class Gpta: # class - 接近MATLAB的function

    def __init__(self, data, log, results): # 函数Gpta的输入：data, log, results
        self.log = log
        self.results = results
        self.data = data
        self.cue = [] # 初始为空
        self.temp = MyTemperature
        self.maxtokens = MyMaxTokens
        self.freqpenalty = MyFreqPenalty
        self.client = OpenAI(api_key = MyAPI)

    def gpt_associations(self, cue, model = MyModel): 
        # 函数gpt_associations的输入：cue, model, 其中model调用时不用定义,
        # 构建给chatGPT输入的消息列表/Prompt
        message = [
            {
                "role": "user", # role - 作为用户调用API
                "content": "我会给你一个词或者字。看到我给你的词或者字以后，"
                + "输入你的脑海里没有经过思考就浮现出来的三个词语或者字,用半角逗号隔开。"
                + "注意，只给出词或字即可，不要解释，不要添加多余信息。"
                + "这一个词或者字是："
                + cue
                + "。", # content - 给chatGPT的指导语
            }
        ]
        
        # 与OpenAI的聊天模型进行交互，需要以下信息：messages
        response = client.chat.completions.create(
            model = model,
            messages = message,
            temperature = self.temp,
            max_tokens = self.maxtokens,
            frequency_penalty = self.freqpenalty,
            stream = False,
        )

        self.cue.append(cue) # self.cue是新建的列表
        
        with open(self.log, "a", encoding = "utf-8") as file: # a - 是写入时追加，w - 是写入时覆盖
            # 将结果写入日志文件self.log
            file.write(
                cue
                + "，，，\n"
                + response.choices[0].message.content
                + "。。。"
                + "\n"
            )

        # 将结果添加到列表中
        self.results.append( # self.results是新建的列表
            {
                "cue": cue,
                "model": model,
                "message": message,
                "response": response,
                "temperature": self.temp,
                "max_tokens": self.maxtokens,
                "frequency_penalty": self.freqpenalty,
            } # {}是创建字典，接近MATLAB的struct, 参数存储入self.results
        )

        # 将结果保存到pickle文件，类似MATLAB的mat文件
        with open(self.data, "wb") as file: # wb - 二进制写入，rb - 二进制读取
            pickle.dump(self.results, file) # 保存self.results到file

        print("获取并保存线索成功：" + cue)
        print("chatGPT的回答是：" + response.choices[0].message.content + "\n")

In [ ]:
# 创建Gpta类的实例, 仅执行了def __init__下的内容
gpta = Gpta(
    data = "gpt4o_SWOWZH_results.pkl",
    log = "gpt4o_SWOWZH_results.txt",
    results=[],
)

# 处理不重复的提示词
k = 0
for i in CUES_all: # for循环
    gpta.gpt_associations(i) # 函数最后没有return,不会返回内容
    k += 1
    if k >= AnswerTime: # 终止条件：chatGPT回答了[AnswerTime]次
        break

# 处理重复的提示词
# currstate = gpta.cue
# for i in range(len(currstate), len(CUES)):
#     gpta.gpt_associations(CUES[i])

获取并保存线索成功：位置
chatGPT的回答是：地点，方向，坐标

获取并保存线索成功：安全
chatGPT的回答是：保障、和平、预防。

获取并保存线索成功：姐妹
chatGPT的回答是：亲密，悄悄话，一起。

获取并保存线索成功：结果
chatGPT的回答是：成功、失败、分析

获取并保存线索成功：物理
chatGPT的回答是：力学，相对，量子

获取并保存线索成功：放松
chatGPT的回答是：休息，假日，轻松

获取并保存线索成功：语言
chatGPT的回答是：沟通，表达，文字

获取并保存线索成功：训练
chatGPT的回答是：重复、努力、成长

获取并保存线索成功：发现
chatGPT的回答是：科学，探索，新奇

获取并保存线索成功：平等
chatGPT的回答是：权利、公正、包容

获取并保存线索成功：地狱
chatGPT的回答是：火焰、孤独、折磨。

获取并保存线索成功：侦探
chatGPT的回答是：案件，推理，线索

获取并保存线索成功：反应
chatGPT的回答是：化学，速度，行动

获取并保存线索成功：描述
chatGPT的回答是：形容，叙述，表达。

获取并保存线索成功：葬礼
chatGPT的回答是：棺木，哀悼，哭泣

获取并保存线索成功：介绍
chatGPT的回答是：开场、自我、描述。

获取并保存线索成功：愚蠢
chatGPT的回答是：错误，愚贪，麻木

获取并保存线索成功：火山
chatGPT的回答是：熔岩，喷发，高温

获取并保存线索成功：合同
chatGPT的回答是：签署、条款、法律

获取并保存线索成功：幽默
chatGPT的回答是：笑话，欢乐，放松

获取并保存线索成功：建筑
chatGPT的回答是：工地,设计,砖块

获取并保存线索成功：暴力
chatGPT的回答是：战争、恐惧、冲突

获取并保存线索成功：啤酒
chatGPT的回答是：泡沫，瓶子，凉爽。

获取并保存线索成功：武器
chatGPT的回答是：战斗、防御、攻击

获取并保存线索成功：老鼠
chatGPT的回答是：米奇，尾巴，洞穴。

获取并保存线索成功：健康
chatGPT的回答是：身体，活力，快乐。

获取并保存线索成功：复仇
chatGPT的回答是：恨意、对抗、赎罪

获取并保存线索成功：卡车
chatGPT的回答是：运输、大型、司机

获取并保存线索成功：宠

获取并保存线索成功：介绍
chatGPT的回答是：书本，见面，背景

获取并保存线索成功：缺席
chatGPT的回答是：会议、空位、遗憾

获取并保存线索成功：爸爸
chatGPT的回答是：妈妈、家庭、安全感

获取并保存线索成功：文章
chatGPT的回答是：阅读、写作、报告

获取并保存线索成功：训练
chatGPT的回答是：跑步、坚持、提升。

获取并保存线索成功：热情
chatGPT的回答是：温暖，活力，阳光

获取并保存线索成功：方法
chatGPT的回答是：技巧，策略，手段

获取并保存线索成功：海盗
chatGPT的回答是：船只、藏宝、航海。

获取并保存线索成功：动物
chatGPT的回答是：狮子，猫咪，大象

获取并保存线索成功：胜利
chatGPT的回答是：喜悦、成就、光荣

获取并保存线索成功：道歉
chatGPT的回答是：原谅，谅解，和好

获取并保存线索成功：节日
chatGPT的回答是：庆祝、假期、欢乐

获取并保存线索成功：行为
chatGPT的回答是：动作、反应、习惯

获取并保存线索成功：旋律
chatGPT的回答是：音乐，动听，节奏

获取并保存线索成功：友谊
chatGPT的回答是：亲情，爱情，缘分

获取并保存线索成功：互动
chatGPT的回答是：交流，沟通，共享

获取并保存线索成功：屁股
chatGPT的回答是：肉，软，大。

获取并保存线索成功：帝国
chatGPT的回答是：权力、领土、衰落

获取并保存线索成功：姐妹
chatGPT的回答是：友情，亲密，支持

获取并保存线索成功：年轻
chatGPT的回答是：青春，活力，未来

获取并保存线索成功：专家
chatGPT的回答是：知识，权威，顾问

获取并保存线索成功：新闻
chatGPT的回答是：资讯，报告，媒体

获取并保存线索成功：感觉
chatGPT的回答是：情绪,知觉,体验

获取并保存线索成功：歌手
chatGPT的回答是：舞台，音乐，麦克风。

获取并保存线索成功：机会
chatGPT的回答是：成功、挑战、抓住

获取并保存线索成功：学生
chatGPT的回答是：课堂，作业，学习

获取并保存线索成功：计算
chatGPT的回答是：机算，数学，科技

获取并保存线索成功：财产
chatGPT的回答是：财富、价值、资本

获取并保存线索成功：目的
chat

获取并保存线索成功：成就
chatGPT的回答是：成功、奋斗、荣耀

获取并保存线索成功：暴力
chatGPT的回答是：战争、伤害、对抗

获取并保存线索成功：椅子
chatGPT的回答是：桌子、人、坐垫

获取并保存线索成功：按摩
chatGPT的回答是：放松，舒缓，压力

获取并保存线索成功：历史
chatGPT的回答是：文化，事件，朝代

获取并保存线索成功：火山
chatGPT的回答是：熔岩、岩浆、喷发

获取并保存线索成功：经济
chatGPT的回答是：财富、增长、市场

获取并保存线索成功：语言
chatGPT的回答是：沟通，文字，表达

获取并保存线索成功：国王
chatGPT的回答是：王冠、宫殿、权力

获取并保存线索成功：屁股
chatGPT的回答是：椅子、大腿、疼

获取并保存线索成功：考试
chatGPT的回答是：紧张、学期、复习

获取并保存线索成功：冲突
chatGPT的回答是：战斗，对立，摩擦

获取并保存线索成功：行为
chatGPT的回答是：模式，反应，决策

获取并保存线索成功：介绍
chatGPT的回答是：导入，引入，推荐

获取并保存线索成功：伤疤
chatGPT的回答是：痛苦、回忆、愈合

获取并保存线索成功：障碍
chatGPT的回答是：困难，阻碍，限制

获取并保存线索成功：竞争者
chatGPT的回答是：对手，比赛，挑战。

获取并保存线索成功：互动
chatGPT的回答是：交流，联系，合作。

获取并保存线索成功：骄傲
chatGPT的回答是：成功、自信、荣耀

获取并保存线索成功：和谐
chatGPT的回答是：平静，融洽，共鸣

获取并保存线索成功：街道
chatGPT的回答是：车流，人群，楼宇

获取并保存线索成功：事件
chatGPT的回答是：新闻，发生，曝光

获取并保存线索成功：计算
chatGPT的回答是：机，数，量

获取并保存线索成功：能量
chatGPT的回答是：太阳，要素，动力。

获取并保存线索成功：宠物
chatGPT的回答是：猫，狗，可爱

获取并保存线索成功：缺席
chatGPT的回答是：空位，怀念，出勤。

获取并保存线索成功：健康
chatGPT的回答是：运动，饮食，睡眠

获取并保存线索成功：男人
chatGPT的回答是：阳刚、坚强、责任

获取并保存线索成功：平等
chatGPT的回

获取并保存线索成功：冲突
chatGPT的回答是：战争，争执，和平。

获取并保存线索成功：森林
chatGPT的回答是：树木，大自然，宁静

获取并保存线索成功：毕业
chatGPT的回答是：仪式、工作、离别

获取并保存线索成功：智慧
chatGPT的回答是：聪明、知识、思考。

获取并保存线索成功：卡车
chatGPT的回答是：大型、运输、道路

获取并保存线索成功：笔记
chatGPT的回答是：纸张、记录、本子

获取并保存线索成功：珠宝
chatGPT的回答是：首饰、金属、奢华

获取并保存线索成功：火山
chatGPT的回答是：岩浆，爆发，熔岩

获取并保存线索成功：海盗
chatGPT的回答是：船，寻宝，海洋

获取并保存线索成功：提示
chatGPT的回答是：密码、信号、标记

获取并保存线索成功：成就
chatGPT的回答是：成功，荣誉，努力。

获取并保存线索成功：考试
chatGPT的回答是：学校、复习、紧张

获取并保存线索成功：玻璃
chatGPT的回答是：透明、窗户、破碎

获取并保存线索成功：语言
chatGPT的回答是：文字，交流，沟通

获取并保存线索成功：老鼠
chatGPT的回答是：猫、窗口、奶酪

获取并保存线索成功：困难
chatGPT的回答是：坚持，勇气，挑战

获取并保存线索成功：专家
chatGPT的回答是：学者，知识，研究。

获取并保存线索成功：水果
chatGPT的回答是：苹果， 香蕉，葡萄

获取并保存线索成功：侦探
chatGPT的回答是：推理，真相，探案

获取并保存线索成功：旋律
chatGPT的回答是：音乐，和弦，动听

获取并保存线索成功：经济
chatGPT的回答是：发展，市场，增长

获取并保存线索成功：压力
chatGPT的回答是：工作、负担、焦虑

获取并保存线索成功：发现
chatGPT的回答是：探索、科学、新知

获取并保存线索成功：骄傲
chatGPT的回答是：自豪、自信、、成功

获取并保存线索成功：吸引
chatGPT的回答是：目光，魅力，磁铁

获取并保存线索成功：革命
chatGPT的回答是：战争、斗争、变化

获取并保存线索成功：能量
chatGPT的回答是：光，动力，热量

获取并保存线索成功：暴力
chatGPT的回答是：战争、伤害、混乱

获取并保存线索成功：决定
cha

获取并保存线索成功：结果
chatGPT的回答是：成功，失败，结局

获取并保存线索成功：珠宝
chatGPT的回答是：金银、钻石、首饰。

获取并保存线索成功：描述
chatGPT的回答是：文字、形容、表达

获取并保存线索成功：我
chatGPT的回答是：你、人、自己

获取并保存线索成功：小说
chatGPT的回答是：故事，角色，虚构

获取并保存线索成功：年龄
chatGPT的回答是：增长，岁月，青春

获取并保存线索成功：笔记
chatGPT的回答是：写字，记录，学习

获取并保存线索成功：神经
chatGPT的回答是：元件，细胞，大脑

获取并保存线索成功：灵魂
chatGPT的回答是：幽深，生命，本质

获取并保存线索成功：骆驼
chatGPT的回答是：沙漠、驼峰、行走

获取并保存线索成功：计算
chatGPT的回答是：数学、程序、分析

获取并保存线索成功：犹豫
chatGPT的回答是：不决、无法、矛盾

获取并保存线索成功：殖民地
chatGPT的回答是：侵略者、资源剥削、本地文化

获取并保存线索成功：愚蠢
chatGPT的回答是：傻瓜，错误，无知

获取并保存线索成功：国王
chatGPT的回答是：皇后、王座、国家

获取并保存线索成功：街道
chatGPT的回答是：行人、店铺、熙熙攘攘。

获取并保存线索成功：丝绸
chatGPT的回答是：滑，韧，柔

获取并保存线索成功：武器
chatGPT的回答是：战争，火炮，刀剑

获取并保存线索成功：早餐
chatGPT的回答是：面包，牛奶，咖啡

获取并保存线索成功：男人
chatGPT的回答是：阳刚，责任，力量

获取并保存线索成功：洋葱
chatGPT的回答是：哭泣、层次、切割

获取并保存线索成功：物理
chatGPT的回答是：科学，能量，力学

获取并保存线索成功：酸
chatGPT的回答是：甜、涩、柠檬

获取并保存线索成功：辐射
chatGPT的回答是：核能，危害，电磁

获取并保存线索成功：艺术
chatGPT的回答是：画，灵感，经典。

获取并保存线索成功：冲突
chatGPT的回答是：矛盾，争执，碰撞。

获取并保存线索成功：发明
chatGPT的回答是：创造，创新，机器

获取并保存线索成功：幽默
chatGPT的回答是：笑声、趣味、开心

获取并保存线索成功：葬礼
chatG

...

获取并保存线索成功：癌症
chatGPT的回答是：绝症、化疗、恐惧.

获取并保存线索成功：介绍
chatGPT的回答是：开头，我们，人选

获取并保存线索成功：爸爸
chatGPT的回答是：妈妈、家、爱。

获取并保存线索成功：独立
chatGPT的回答是：自主，自由，自信。

获取并保存线索成功：和谐
chatGPT的回答是：共鸣，默契，融洽

获取并保存线索成功：新闻
chatGPT的回答是：报道，时事，媒体。

获取并保存线索成功：结果
chatGPT的回答是：尾声，收获，答案

获取并保存线索成功：方法
chatGPT的回答是：步骤、技巧、解决方案

获取并保存线索成功：经济
chatGPT的回答是：发展、增长、市场

获取并保存线索成功：合同
chatGPT的回答是：工作，签字，法律

获取并保存线索成功：压力
chatGPT的回答是：工作、紧张、疲惫。

获取并保存线索成功：友谊
chatGPT的回答是：真诚，友情，共同。

获取并保存线索成功：神经
chatGPT的回答是：反应，中枢，连接

获取并保存线索成功：历史
chatGPT的回答是：滚滚, 兴衰, 时间

获取并保存线索成功：孩子
chatGPT的回答是：童年、家庭、游戏

获取并保存线索成功：帝国
chatGPT的回答是：皇帝、权力、霸主

获取并保存线索成功：喜剧
chatGPT的回答是：笑声、演员、电影

获取并保存线索成功：衣服
chatGPT的回答是：穿搭，布料，时尚。

获取并保存线索成功：夜晚
chatGPT的回答是：星空，宁静，黑暗。

获取并保存线索成功：照片
chatGPT的回答是：光影，记忆，记录

获取并保存线索成功：物理
chatGPT的回答是：能量，定律，实验。

获取并保存线索成功：笔记
chatGPT的回答是：纸质，想法，记录。

获取并保存线索成功：姐妹
chatGPT的回答是：朋友，关系，亲情

获取并保存线索成功：行为
chatGPT的回答是：态度，规范，举止

获取并保存线索成功：旋律
chatGPT的回答是：音乐，声音，节拍。

获取并保存线索成功：幽默
chatGPT的回答是：笑声，风趣，交流。

获取并保存线索成功：民主
chatGPT的回答是：平等、自由、人权

获取并保存线索成功：侦探
chatGPT的回答是：福尔摩斯、推理、犯罪

获取并保

获取并保存线索成功：对手
chatGPT的回答是：竞争，挑战，胜负

获取并保存线索成功：同情
chatGPT的回答是：理解，关怀，帮助

获取并保存线索成功：宠物
chatGPT的回答是：猫，狗，可爱

获取并保存线索成功：犹豫
chatGPT的回答是：迟疑、不安、决定

获取并保存线索成功：灾难
chatGPT的回答是：地震、洪水、损失。

获取并保存线索成功：玻璃
chatGPT的回答是：透明、碎片、窗户

获取并保存线索成功：合同
chatGPT的回答是：文件，签字，法律

获取并保存线索成功：尸体
chatGPT的回答是：死亡，冷冰，葬礼

获取并保存线索成功：感觉
chatGPT的回答是：情绪,直觉,触觉

获取并保存线索成功：卡车
chatGPT的回答是：司机、公路、货物

获取并保存线索成功：能量
chatGPT的回答是：光，热，活力

获取并保存线索成功：人类
chatGPT的回答是：社会，文明，未来。

获取并保存线索成功：小说
chatGPT的回答是：读者、故事、情节

获取并保存线索成功：我
chatGPT的回答是：你，他，人

获取并保存线索成功：数字
chatGPT的回答是：计算、科技、网络

获取并保存线索成功：警察
chatGPT的回答是：制服、执法、正义

获取并保存线索成功：影子
chatGPT的回答是：黑暗、阳光、形状

获取并保存线索成功：酸
chatGPT的回答是：甜、苦、涩。

获取并保存线索成功：友谊
chatGPT的回答是：友情，信任，伙伴

获取并保存线索成功：艺术
chatGPT的回答是：创作、表现、美丽

获取并保存线索成功：下雨
chatGPT的回答是：天凉，打伞，好玩

获取并保存线索成功：笔记
chatGPT的回答是：书本、记录、笔迹

获取并保存线索成功：早餐
chatGPT的回答是：牛奶，面包，鸡蛋

获取并保存线索成功：事件
chatGPT的回答是：新闻，影响，发展。

获取并保存线索成功：学生
chatGPT的回答是：老师，校园，书本

获取并保存线索成功：毕业
chatGPT的回答是：求职，纪念册，礼服

获取并保存线索成功：文章
chatGPT的回答是：句子、书籍、文学

获取并保存线索成功：收入
chatGPT的回答是：支出，工资，财务

获取并保存线索成功：爸爸
chatGPT的回答是：妈妈

获取并保存线索成功：照片
chatGPT的回答是：画面，记忆，记录

获取并保存线索成功：丝绸
chatGPT的回答是：顺滑，织物，中国

获取并保存线索成功：障碍
chatGPT的回答是：困难，距离，阻挡

获取并保存线索成功：电梯
chatGPT的回答是：上下、按钮、楼层

获取并保存线索成功：发明
chatGPT的回答是：创造、创新、科学

获取并保存线索成功：提示
chatGPT的回答是：消息、方向、内容

获取并保存线索成功：数字
chatGPT的回答是：一零一, 科技, 运算

获取并保存线索成功：艺术
chatGPT的回答是：创意、美丽、表达

获取并保存线索成功：小狗
chatGPT的回答是：可爱，毛茸茸，活泼

获取并保存线索成功：人类
chatGPT的回答是：地球，文明，进化

获取并保存线索成功：吸引
chatGPT的回答是：目光，魅力，注意。

获取并保存线索成功：收入
chatGPT的回答是：工资、财富、支出。

获取并保存线索成功：感觉
chatGPT的回答是：触觉,思维,感官

获取并保存线索成功：面包
chatGPT的回答是：黄油、早餐、烤箱

获取并保存线索成功：辐射
chatGPT的回答是：核, 能量, 危险

获取并保存线索成功：建筑
chatGPT的回答是：楼房，工地，设计

获取并保存线索成功：同意
chatGPT的回答是：接受，认同，批准。

获取并保存线索成功：行为
chatGPT的回答是：动作，表现，方式。

获取并保存线索成功：国王
chatGPT的回答是：皇后、王国、皇冠

获取并保存线索成功：语言
chatGPT的回答是：交流，文字，文化。

获取并保存线索成功：噪音
chatGPT的回答是：声音，吵闹，干扰。

获取并保存线索成功：歌手
chatGPT的回答是：音乐，演唱，舞台

获取并保存线索成功：孤独
chatGPT的回答是：寂寞，空虚，落寞

获取并保存线索成功：平等
chatGPT的回答是：公平、公正、和谐

获取并保存线索成功：合同
chatGPT的回答是：签署，备注，违约

获取并保存线索成功：报纸
chatGPT的回答是：新闻、纸张、时事

获取并保存线索成功：教育
chatGPT的回答是：学习，知识，成长

获取并保存线索成功：孩子
chatGPT的回答是：玩具、学校、成长

获取并保存线索成功

获取并保存线索成功：专家
chatGPT的回答是：学者，权威，知识。

获取并保存线索成功：安全
chatGPT的回答是：保护、保障、平和

获取并保存线索成功：酸
chatGPT的回答是：柠檬、醋、味觉

获取并保存线索成功：海盗
chatGPT的回答是：船长、宝藏、海洋

获取并保存线索成功：帝国
chatGPT的回答是：国家、统治者、历史

获取并保存线索成功：椅子
chatGPT的回答是：知道、力量、坐下。

获取并保存线索成功：行为
chatGPT的回答是：思想, 态度, 习惯

获取并保存线索成功：蔬菜
chatGPT的回答是：水果，健康，绿色

获取并保存线索成功：愚蠢
chatGPT的回答是：笨拙，幼稚，糊涂

获取并保存线索成功：绳子
chatGPT的回答是：捆绑，结，长

获取并保存线索成功：学生
chatGPT的回答是：作业、课堂、课本

获取并保存线索成功：孩子
chatGPT的回答是：玩具，欢乐，成长

获取并保存线索成功：照片
chatGPT的回答是：相机，记忆，风景

获取并保存线索成功：食物
chatGPT的回答是：米饭，水果，烹饪

获取并保存线索成功：成就
chatGPT的回答是：努力，奋斗，成功

获取并保存线索成功：年龄
chatGPT的回答是：成长，数字，岁月。

获取并保存线索成功：年轻
chatGPT的回答是：活力，未来，无畏

获取并保存线索成功：灾难
chatGPT的回答是：痛苦，破坏，重建。

获取并保存线索成功：文章
chatGPT的回答是：文采、文字、修辞

获取并保存线索成功：啤酒
chatGPT的回答是：酒吧，慕尼黑，节日。

获取并保存线索成功：笔记
chatGPT的回答是：纸张、记录、书写。

获取并保存线索成功：脾气
chatGPT的回答是：情绪、愤怒、性格。

获取并保存线索成功：机会
chatGPT的回答是：未来、成长、选择。

获取并保存线索成功：幽默
chatGPT的回答是：笑声，风趣，快乐

获取并保存线索成功：丝绸
chatGPT的回答是：顺滑、古代、路线

获取并保存线索成功：邮件
chatGPT的回答是：信箱、信封、电子。

获取并保存线索成功：骄傲
chatGPT的回答是：自信、鲜花、成功

获取并保存线索成功：能量
chatGPT的回答是：激活，动力，火焰。

获取并保存线索成功：

获取并保存线索成功：幽默
chatGPT的回答是：笑声，轻松，快乐。

获取并保存线索成功：人类
chatGPT的回答是：历史，智慧，情感。

获取并保存线索成功：方法
chatGPT的回答是：途径、技巧、步骤

获取并保存线索成功：侦探
chatGPT的回答是：福尔摩斯，推理，谜题

获取并保存线索成功：海盗
chatGPT的回答是：三叉戟、宝藏、船长

获取并保存线索成功：椅子
chatGPT的回答是：桌子、坐、靠背

获取并保存线索成功：牛奶
chatGPT的回答是：早餐，健康，白色

获取并保存线索成功：森林
chatGPT的回答是：树木,绿叶,漫步

获取并保存线索成功：同情
chatGPT的回答是：怜悯，帮忙，共情。

获取并保存线索成功：民主
chatGPT的回答是：自由、平等、选举

获取并保存线索成功：愚蠢
chatGPT的回答是：无知，错误，荒谬

